# Unexecuted reviewed DQN variant

Target mask uses true termination; episode stopping still uses terminated OR truncated. All outputs cleared. No revised scores exist. Historical scores belong only to notebook.ipynb.


# Deep Q-Network (DQN)

* Full Name: BEHRAD MAZAHERI
* Student Number:[student ID removed]

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque, namedtuple
import random
import matplotlib.pyplot as plt
import gymnasium as gym

In [ ]:
# MP4 Libraries
from IPython.display import HTML
from base64 import b64encode
import cv2
import numpy as np

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


### Helper Functions for Video

Function to convert MP4 to displayable HTML5 video and display

In [ ]:
def show_video(video_path, width=480):
    with open(video_path, "rb") as f:
        video_data = f.read()
    encoded = b64encode(video_data).decode("utf-8")
    html = f'''
        <video width="{width}" controls>
            <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
        </video>
    '''
    return HTML(html)

# def save_agent_video(env, agent, filename="agent_demo.mp4"):
#     frames = []
#     state, _ = env.reset()
#     done = False

#     while not done:
#         action = agent.select_action(state)
#         next_state, reward, terminated, truncated, info = env.step(action)
#         done = terminated or truncated
#         frames.append(env.render())
#         state = next_state

#     height, width, layers = frames[0].shape
#     out = cv2.VideoWriter(
#         filename, cv2.VideoWriter_fourcc(*"mp4v"), 30, (width, height)
#     )
#     for f in frames:
#         out.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
#     out.release()
def save_agent_video(env_name, agent, filename="agent_demo.mp4", seed=42, max_steps=1000):
    import cv2
    import gymnasium as gym

    # Important: create a new environment with render_mode="rgb_array"
    env = gym.make(env_name, render_mode="rgb_array")

    frames = []
    state, _ = env.reset(seed=seed)
    done = False
    total_reward = 0.0

    for t in range(max_steps):
        # Select action without exploration if possible
        try:
            action = agent.select_action(state, training=False)
        except TypeError:
            action = agent.select_action(state)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        frame = env.render()

        if frame is not None:
            frames.append(frame)

        total_reward += reward
        state = next_state

        if done:
            break

    env.close()

    if len(frames) == 0:
        raise ValueError(
            "No frames were rendered. Make sure env is created with render_mode='rgb_array'."
        )

    height, width, layers = frames[0].shape

    out = cv2.VideoWriter(
        filename,
        cv2.VideoWriter_fourcc(*"mp4v"),
        30,
        (width, height)
    )

    for frame in frames:
        out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

    out.release()

    print("Saved video to:", filename)
    print("Episode length:", t + 1)
    print("Total reward:", total_reward)

    return filename

### Checkout the Environment


In [ ]:
# TODO
# Print the environment action space dim
# Print the environment observation space dim
# Do a random rollout and visualize the environment

# Create environment for checking spaces

env = gym.make("LunarLander-v3", render_mode="rgb_array")
print("Action space:", env.action_space)
print("Number of actions:", env.action_space.n)
print("Observation space:", env.observation_space)
print("Observation dimension:", env.observation_space.shape[0])

# Reset environment
state, info = env.reset(seed=42)

print("\nInitial state:")
print(state)
print("State shape:", state.shape)

# Take one random action
action = env.action_space.sample()
next_state, reward, terminated, truncated, info = env.step(action)
done = terminated or truncated

print("\nOne random step:")
print("Action:", action)
print("Next state shape:", next_state.shape)
print("Reward:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("Done:", done)

env.close()

def save_random_video(env_name="LunarLander-v3", filename="random_rollout.mp4", seed=42, max_steps=1000):
    env = gym.make(env_name, render_mode="rgb_array")
    
    frames = []
    state, info = env.reset(seed=seed)
    total_reward = 0.0
    
    for t in range(max_steps):
        # Select random action
        action = env.action_space.sample()
        
        # Step in environment
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        
        # Render frame
        frame = env.render()
        frames.append(frame)
        
        total_reward += reward
        state = next_state
        
        if done:
            break
    
    env.close()
    
    # Save frames as mp4
    height, width, layers = frames[0].shape
    
    out = cv2.VideoWriter(
        filename,
        cv2.VideoWriter_fourcc(*"mp4v"),
        30,
        (width, height)
    )
    
    for frame in frames:
        # OpenCV expects BGR, Gym gives RGB
        out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    
    out.release()
    
    print("Saved video to:", filename)
    print("Episode length:", t + 1)
    print("Total reward:", total_reward)
    
    return filename


video_path = save_random_video(
    env_name="LunarLander-v3",
    filename="random_rollout.mp4",
    seed=42,
    max_steps=1000
)

show_video(video_path)




### Replay Buffer
In DQN, the agent learns from past experiences.  
This buffer stores transitions in the form of (state, action, reward, next_state, done).

+ Your task: complete the `push` and `sample` methods.

- `push` should add a new Transition into the buffer.
- `sample` should randomly return a batch of transitions  
  (this helps break correlations and stabilize learning).

In [ ]:
# TODO
from collections import deque, namedtuple
import random

Transition = namedtuple(
    'Transition', 
    ['state', 'action', 'reward', 'next_state', 'done']
)

class ReplayBuffer:
    """Stores experiences for experience replay"""
    
    def __init__(self, capacity: int = 100):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)
    
    def push(self, *args):
        """
        Add a new transition to the buffer.
        
        Expected args:
        state, action, reward, next_state, done
        """
        transition = Transition(*args)
        self.buffer.append(transition)
    
    def sample(self, batch_size: int) -> Transition:
        """
        Randomly sample a batch of transitions.
        
        Returns a Transition where each field is a tuple:
        batch.state, batch.action, batch.reward, batch.next_state, batch.done
        """
        transitions = random.sample(self.buffer, batch_size)
        batch = Transition(*zip(*transitions))
        return batch
    
    def __len__(self):
        return len(self.buffer)

### Q-Network

In [ ]:
class QNetwork(nn.Module):
    """Simple neural network to approximate Q(s,a)"""
    
    def __init__(self, state_dim: int, action_dim: int):
        super(QNetwork, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )
    
    def forward(self, x):
        return self.network(x)

###  Hyperparameters of the DQN Agent
These values control exploration, learning stability, and update frequency.
Feel free to experiment — small changes can significantly affect performance.

- `gamma`: discount factor  
- `epsilon`: initial exploration rate  
- `epsilon_decay`: how fast the agent shifts from exploration to exploitation  
- `batch_size`: number of samples from ReplayBuffer for each update  
- `target_update_freq`: how often the target network is updated


### Action Selection (Epsilon-Greedy)

The agent chooses between:
- exploration (random action) with probability epsilon  
- exploitation (best predicted action) otherwise


### Update Step of DQN

This is the heart of the algorithm:
1. Sample a batch from ReplayBuffer  
2. Compute predicted Q-values (current network)  
3. Compute target Q-values (target network)  
4. Calculate loss  
5. Optimize network weights  
6. Periodically update the target network

+ Your task: fill in the missing steps and complete the TODOs.
Follow the comments carefully — they guide the flow of DQN training.


In [ ]:
import torch.nn.functional as F
class DQNAgent:
    """
    Simple DQN Agent
    
    Key components:
    - Q-Network: approximates Q(s,a)
    - Target Network: provides stable targets for learning
    - Replay Buffer: stores past experiences
    - Epsilon-greedy: balances exploration/exploitation
    """
    
    def __init__(self, state_dim: int, action_dim: int):
        # Hyperparameters
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.9998
        self.batch_size = 64
        self.target_update_freq = 250
        self.action_dim = action_dim
        
        # Networks
        self.q_network = QNetwork(state_dim, action_dim).to(device)
        self.target_network = QNetwork(state_dim, action_dim).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()
        
        # Optimizer and memory
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=3e-4)
        self.memory = ReplayBuffer(capacity=100000)
        self.steps = 0
    
    def select_action(self, state, training=True):
        """
        Select an action using epsilon-greedy policy.
        """
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        
        with torch.no_grad():
            q_values = self.q_network(state_tensor)
        
        return q_values.argmax(dim=1).item()
    
    def store_transition(self, state, action, reward, next_state, done):
        """
        Store one transition in replay buffer.
        """
        self.memory.push(state, action, reward, next_state, done)
    
    def update(self):
        """
        One DQN learning update.
        """
        if len(self.memory) < self.batch_size:
            return None
        
        # Sample batch
        batch = self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(np.array(batch.state)).to(device)
        actions = torch.LongTensor(batch.action).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(batch.reward).to(device)
        next_states = torch.FloatTensor(np.array(batch.next_state)).to(device)
        dones = torch.FloatTensor(batch.done).to(device)
        
        # Current Q-values: Q(s,a; theta)
        current_q_values = self.q_network(states).gather(1, actions).squeeze(1)
        
        # Target Q-values: r + gamma * max_a' Q_target(s', a')
        with torch.no_grad():
            next_q_values = self.target_network(next_states).max(dim=1)[0]
            target_q_values = rewards + self.gamma * next_q_values * (1 - dones)
        
        # Loss
        loss = F.mse_loss(current_q_values, target_q_values)
        
        # Optimize online network
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=10.0)

        self.optimizer.step()
        
        # Update target network periodically
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
            self.epsilon = max(self.epsilon, self.epsilon_min)
        
        return loss.item()

### Training Loop

This function orchestrates the entire training process:
- Runs episodes
- Tracks scores and losses
- Updates the agent at each timestep
- Prints progress

+ Your task: you only need to complete the parts marked with TODO.
After that, run the notebook to train your agent!


In [ ]:
def train_dqn(env, agent, num_episodes=1000, max_steps=1000):
    scores = []
    losses = []
    
    for episode in range(num_episodes):
        state, _ = env.reset(seed=SEED + episode)
        score = 0
        
        episode_losses = []
        
        for step in range(max_steps):
            # Select action using epsilon-greedy policy
            action = agent.select_action(state, training=True)
            
            # Take action in the environment
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Store transition in replay buffer
            agent.store_transition(state, action, reward, next_state, terminated)  # bootstrap at external time limits
            
            # Update the agent
            loss = agent.update()
            if loss is not None:
                episode_losses.append(loss)
            
            # Accumulate reward
            score += reward
            
            # Move to next state
            state = next_state
            
            if done:
                break
        
        scores.append(score)
        
        if len(episode_losses) > 0:
            losses.append(np.mean(episode_losses))
        else:
            losses.append(0)
        
        if (episode + 1) % 50 == 0:
            avg = np.mean(scores[-50:])
            print(
                f"Episode {episode+1}/{num_episodes} | "
                f"Score: {score:.1f} | "
                f"Avg: {avg:.1f} | "
                f"Loss: {losses[-1]:.4f} | "
                f"ε: {agent.epsilon:.3f}"
            )
    
    return scores, losses   

In [ ]:
print("="*60)
print("TRAINING SIMPLE DQN ON LUNAR LANDER")
print("="*60)

env = gym.make('LunarLander-v3')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

print(f"State dim: {state_dim}, Action dim: {action_dim}\n")

# Create agent
dqn_agent = DQNAgent(state_dim, action_dim)

# Train
dqn_scores, dqn_losses = train_dqn(
    env,
    dqn_agent,
    num_episodes=1000,
    max_steps=1000
)

env.close()
# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Video
# video_dqn = save_agent_video(env, dqn_agent, filename="dqn.mp4")
video_dqn = save_agent_video(
    env_name="LunarLander-v3",
    agent=dqn_agent,
    filename="dqn.mp4",
    seed=42
)

axes[0].plot(dqn_scores)
axes[0].plot(np.convolve(dqn_scores, np.ones(50)/50, mode='valid'), 'r-', linewidth=2, label='Moving Avg')
axes[0].axhline(y=200, color='g', linestyle='--', label='Target')
axes[0].set_title('DQN - Scores')
axes[0].set_xlabel('Episode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(dqn_losses)
axes[1].set_title('DQN - Loss')
axes[1].set_xlabel('Episode')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDQN Final Avg Score (last 50): {np.mean(dqn_scores[-50:]):.1f}")

show_video(video_dqn)

# Double DQN (DDQN)

### The Problem We're Solving
Regular DQN uses the **same network** to pick the best action AND to evaluate it.
This causes **overestimation** — the agent thinks actions are better than they really are.

Think of it like grading your own test: you're probably too nice to yourself!

### The Solution: Split the Jobs
Double DQN uses **two networks**:
- **Online Network** → picks the best action (the student)
- **Target Network** → evaluates that action's value (the teacher)


### Hyperparameters
Same as DQN — no new hyperparameters needed!
- `gamma`: discount factor
- `epsilon`: exploration rate
- `epsilon_decay`: how fast we stop exploring
- `batch_size`: samples per update
- `target_update_freq`: how often to sync networks

### The Double DQN Update Step
This is where the magic happens:

1. Sample a batch from ReplayBuffer
2. Get current Q-values from **online network**
3. Find **best action** using online network  $\to a_\text{best} = \argmax Q_\text{online}(s')$
4. Get Q-values for those actions from **target network**  $\to Q_\text{target}(s', a_\text{best})$
5. Compute TD target: $y = r + \gamma \times Q_\text{target}(s', a_\text{best})$
6. Compute loss between online predictions and target
7. Optimize online network
8. Periodically update target network

In [ ]:
class DDQNAgent(DQNAgent):
    """
    Double DQN Agent
    
    Difference from DQN:
    - Online network selects the best next action
    - Target network evaluates that selected action
    """
    def update(self):
        """
        One Double DQN learning update.
        """
        if len(self.memory) < self.batch_size:
            return None
        
        # Sample batch
        batch = self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(np.array(batch.state)).to(device)
        actions = torch.LongTensor(batch.action).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(batch.reward).to(device)
        next_states = torch.FloatTensor(np.array(batch.next_state)).to(device)
        dones = torch.FloatTensor(batch.done).to(device)
        
        # Current Q-values: Q_online(s, a; theta)
        current_q_values = self.q_network(states).gather(1, actions).squeeze(1)
        
        # Double DQN target
        with torch.no_grad():
            # 1. Online network selects the best next action
            next_actions = self.q_network(next_states).argmax(dim=1, keepdim=True)
            
            # 2. Target network evaluates that selected action
            next_q_values = self.target_network(next_states).gather(1, next_actions).squeeze(1)
            
            # 3. Bellman target
            target_q_values = rewards + self.gamma * next_q_values * (1 - dones)
        
        # Loss
        loss = F.mse_loss(current_q_values, target_q_values)
        
        # Optimize online network
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=10.0)
        self.optimizer.step()
        
        # Update target network periodically
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
            self.epsilon = max(self.epsilon, self.epsilon_min)
        
        return loss.item()

In [ ]:
print("="*60)
print("TRAINING DOUBLE DQN ON LUNAR LANDER")
print("="*60)

# Reset seeds for fair comparison
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

env = gym.make("LunarLander-v3")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

print(f"State dim: {state_dim}, Action dim: {action_dim}\n")

ddqn_agent = DDQNAgent(state_dim, action_dim)

ddqn_scores, ddqn_losses = train_dqn(
    env,
    ddqn_agent,
    num_episodes=1000,
    max_steps=1000
)

env.close()
# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Video
# video_ddqn = save_agent_video(env2, ddqn_agent, filename="ddqn.mp4")

video_ddqn = save_agent_video(
    env_name="LunarLander-v3",
    agent=ddqn_agent,
    filename="ddqn.mp4",
    seed=42
)


axes[0].plot(ddqn_scores, alpha=0.5, label='DDQN')
axes[0].plot(np.convolve(ddqn_scores, np.ones(50)/50, mode='valid'), 'r-', linewidth=2, label='Moving Avg')
axes[0].axhline(y=200, color='g', linestyle='--', label='Target')
axes[0].set_title('Double DQN - Scores')
axes[0].set_xlabel('Episode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ddqn_losses)
axes[1].set_title('Double DQN - Loss')
axes[1].set_xlabel('Episode')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDDQN Final Avg Score (last 50): {np.mean(ddqn_scores[-50:]):.1f}")

# Dueling DQN

### DuelingQNetwork — Splitting Value and Advantage

Instead of estimating $Q(s,a)$ directly, we break it into two parts:


- **$V(s)$** = How good is it to be in this state? (regardless of action)
- **$A(s,a)$** = How much better is action $a$ compared to other actions?

### Why Is This Better?
Imagine you're watching a movie:
- Some scenes are exciting regardless of what happens → **$V(s)$** is high
- The specific action matters more in tense scenes → **$A(s,a)$** carries more weight

By separating them, the network can learn **$V(s)$** directly without averaging across all actions.



### The Formula
We don't use $Q = V + A$ directly. Instead:

$$ Q(s,a) = V(s) + A(s,a) - mean(A(s,:))$$

In [ ]:
# =============================================================================
# Dueling DQN
# Architecture that separates value and advantage estimation
# =============================================================================
class DuelingQNetwork(nn.Module):
    """
    Dueling Q-Network
    
    It decomposes Q(s,a) into:
    - V(s): value of the state
    - A(s,a): advantage of each action
    
    Then combines them as:
    Q(s,a) = V(s) + A(s,a) - mean_a A(s,a)
    """
    
    def __init__(self, state_dim: int, action_dim: int):
        super().__init__()
        
        self.feature_layer = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )
        
        self.value_stream = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
        
        self.advantage_stream = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )
    
    def forward(self, x):
        features = self.feature_layer(x)
        
        value = self.value_stream(features)
        advantage = self.advantage_stream(features)
        
        q_values = value + advantage - advantage.mean(dim=1, keepdim=True)
        
        return q_values

In [ ]:
class DuelingDQNAgent:
    """
    Dueling DQN uses the dueling network architecture.
    The update rule is the same as DDQN, 
    but the network architecture is different.
    """
    
    def __init__(self, state_dim: int, action_dim: int):
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.9997
        self.batch_size = 64
        self.target_update_freq = 250
        self.action_dim = action_dim
        
        # Networks (Dueling architecture)
        self.q_network = DuelingQNetwork(state_dim, action_dim).to(device)
        self.target_network = DuelingQNetwork(state_dim, action_dim).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()
        
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=5e-4)
        self.memory = ReplayBuffer(capacity=100000)
        self.steps = 0
    
    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        
        with torch.no_grad():
            q_values = self.q_network(state_tensor)
        
        return q_values.argmax(dim=1).item()
    
    def store_transition(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)
    
    def update(self):
        if len(self.memory) < self.batch_size:
            return None
        
        batch = self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(np.array(batch.state)).to(device)
        actions = torch.LongTensor(batch.action).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(batch.reward).to(device)
        next_states = torch.FloatTensor(np.array(batch.next_state)).to(device)
        dones = torch.FloatTensor(batch.done).to(device)
        
        # Current Q values: Q_online(s,a)
        current_q = self.q_network(states).gather(1, actions).squeeze(1)
        
        # Target Q values: Double DQN style
        with torch.no_grad():
            # Online network selects best next action
            next_actions = self.q_network(next_states).argmax(dim=1, keepdim=True)
            
            # Target network evaluates that action
            next_q = self.target_network(next_states).gather(1, next_actions).squeeze(1)
            
            # Bellman target
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        # Compute loss
        loss = nn.MSELoss()(current_q, target_q)
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=10.0)
        self.optimizer.step()
        
        # Update target network
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        
        return loss.item()

In [ ]:
print("="*60)
print("TRAINING DUELING DQN ON LUNAR LANDER")
print("="*60)

env3 = gym.make("LunarLander-v3")
state_dim = env3.observation_space.shape[0]
action_dim = env3.action_space.n

dueling_agent = DuelingDQNAgent(state_dim, action_dim)

dueling_scores, dueling_losses = train_dqn(
    env3,
    dueling_agent,
    num_episodes=1000,
    max_steps=1000
)

env3.close()
# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Video
video_duelingdqn = save_agent_video(
    env_name="LunarLander-v3",
    agent=dueling_agent,
    filename="duelingdqn.mp4",
    seed=42
)


axes[0].plot(dueling_scores)
axes[0].plot(np.convolve(dueling_scores, np.ones(50)/50, mode='valid'), 'r-', linewidth=2, label='Moving Avg')
axes[0].axhline(y=200, color='g', linestyle='--', label='Target')
axes[0].set_title('Dueling DQN - Scores')
axes[0].set_xlabel('Episode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(dueling_losses)
axes[1].set_title('Dueling DQN - Loss')
axes[1].set_xlabel('Episode')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nDueling DQN Final Avg Score (last 50): {np.mean(dueling_scores[-50:]):.1f}")

show_video(video_duelingdqn, width=600)

# Prioritized Experience Replay (PER)

### SumTree — The Priority Organizer

### Why Do We Need It?
Regular replay buffer picks experiences **randomly**.
But some experiences teach us more than others!

Experiences with **high TD error** (big surprise) are more valuable.
We want to pick those more often.

### How Does SumTree Work?
Think of it like a **binary tree** where:
- Each **leaf** = one experience with a priority score
- Each **internal node** = sum of its children


In [ ]:
# =============================================================================
# Prioritized Experience Replay (PER) DQN
# Based on "Prioritized Experience Replay" (Schaul et al., 2016)
# =============================================================================

class SumTree:
    """
    SumTree for Prioritized Experience Replay.
    
    Leaves store priorities.
    Internal nodes store sums of children.
    Root stores total priority.
    """
    
    def __init__(self, capacity: int):
        self.capacity = capacity
        
        # Binary tree:
        # Number of nodes = 2 * capacity - 1
        self.tree = np.zeros(2 * capacity - 1, dtype=np.float32)
        
        # Actual stored transitions
        self.data = np.empty(capacity, dtype=object)
        
        self.write_idx = 0
        self.size = 0
    
    @property
    def total_priority(self):
        return self.tree[0]
    
    def add(self, priority: float, transition: Transition):
        """
        Add transition with given priority.
        """
        # Leaf index corresponding to current write position
        tree_idx = self.write_idx + self.capacity - 1
        
        # Store transition
        self.data[self.write_idx] = transition
        
        # Update tree priority
        self.update(tree_idx, priority)
        
        # Move write pointer circularly
        self.write_idx = (self.write_idx + 1) % self.capacity
        
        # Update size
        self.size = min(self.size + 1, self.capacity)
    
    def update(self, tree_idx: int, priority: float):
        """
        Update priority of a leaf and propagate the change upward.
        """
        change = priority - self.tree[tree_idx]
        self.tree[tree_idx] = priority
        
        # Propagate change to parents
        parent = (tree_idx - 1) // 2
        
        while True:
            self.tree[parent] += change
            
            if parent == 0:
                break
            
            parent = (parent - 1) // 2
    
    def get_leaf(self, value: float):
        """
        Find the leaf whose cumulative priority interval contains value.
        
        Returns:
        tree_idx, priority, transition
        """
        parent_idx = 0
        
        while True:
            left_child = 2 * parent_idx + 1
            right_child = left_child + 1
            
            # If left child is out of bounds, we are at a leaf
            if left_child >= len(self.tree):
                leaf_idx = parent_idx
                break
            
            if value <= self.tree[left_child]:
                parent_idx = left_child
            else:
                value -= self.tree[left_child]
                parent_idx = right_child
        
        data_idx = leaf_idx - self.capacity + 1
        
        return leaf_idx, self.tree[leaf_idx], self.data[data_idx]
    
    def sample(self, batch_size: int):
        """
        Stratified sampling from the priority distribution.
        """
        batch = []
        indices = []
        priorities = []
        
        segment = self.total_priority / batch_size
        
        for i in range(batch_size):
            a = segment * i
            b = segment * (i + 1)
            
            value = random.uniform(a, b)
            
            idx, priority, transition = self.get_leaf(value)
            
            batch.append(transition)
            indices.append(idx)
            priorities.append(priority)
        
        return indices, priorities, batch
    
    def __len__(self):
        return self.size

### The Problem with Uniform Sampling
Regular buffer treats all experiences equally.
But if you already know something well, reviewing it is a waste of time!

### The Solution: Priority-Based Sampling
We sample experiences with probability proportional to their TD error:

+ ${P}(\text{experience}) \propto  |\text{TD error}|^\alpha $

- $\alpha = 0 \to$ uniform sampling (like regular buffer)
- $\alpha = 1 \to$ full prioritization (only important experiences)

### Importance Sampling (IS) Weights
Prioritization changes the distribution → introduces **bias**.
We fix this with IS weights:
+ $w = (N × \mathbb{P}(i))^{-\beta}$

- $\beta = 0 \to$ no correction (fast but biased)
- $\beta = 1 \to$ full correction (unbiased but higher variance)


In [ ]:
class PrioritizedReplayBuffer:
    """
    Prioritized Experience Replay Buffer.
    
    Samples transitions with probability proportional to priority.
    Uses importance sampling weights to correct bias.
    """
    
    def __init__(self, capacity: int = 100000, alpha: float = 0.6, beta: float = 0.4):
        self.capacity = capacity
        self.alpha = alpha
        self.beta = beta
        self.beta_increment = 1e-4
        self.epsilon = 1e-5
        
        self.tree = SumTree(capacity)
        self.max_priority = 1.0
    
    def push(self, *args):
        """
        Add new experience with max priority so it is sampled soon.
        """
        transition = Transition(*args)
        
        priority = self.max_priority
        
        self.tree.add(priority, transition)
    
    def sample(self, batch_size: int):
        indices, priorities, transitions = self.tree.sample(batch_size)

        # Remove any accidental None transitions
        valid = [
            (idx, p, tr)
            for idx, p, tr in zip(indices, priorities, transitions)
            if tr is not None and p > 0
        ]

        # If something went wrong, resample until we have enough valid samples
        while len(valid) < batch_size:
            extra_indices, extra_priorities, extra_transitions = self.tree.sample(batch_size - len(valid))
            for idx, p, tr in zip(extra_indices, extra_priorities, extra_transitions):
                if tr is not None and p > 0:
                    valid.append((idx, p, tr))

        indices, priorities, transitions = zip(*valid)

        priorities = np.array(priorities, dtype=np.float32)

        total_priority = self.tree.total_priority
        probs = priorities / total_priority

        # Numerical stability
        probs = np.maximum(probs, 1e-8)

        weights = (self.tree.size * probs) ** (-self.beta)
        weights = weights / weights.max()
        weights = weights.astype(np.float32)

        batch = Transition(*zip(*transitions))

        states = np.array(batch.state)
        actions = np.array(batch.action)
        rewards = np.array(batch.reward, dtype=np.float32)
        next_states = np.array(batch.next_state)
        dones = np.array(batch.done, dtype=np.float32)

        return states, actions, rewards, next_states, dones, weights, indices
    def update_priorities(self, indices: list, td_errors: np.ndarray):
        """
        Update priorities based on new TD errors.
        """
        for idx, error in zip(indices, td_errors):
            priority = (abs(error) + self.epsilon) ** self.alpha
            
            self.tree.update(idx, priority)
            self.max_priority = max(self.max_priority, priority)
    
    def anneal_beta(self):
        """
        Slowly increase beta toward 1.
        """
        self.beta = min(1.0, self.beta + self.beta_increment)
    
    def __len__(self):
        return len(self.tree)

In [ ]:
class PERDQNAgent:
    """
    DQN with Prioritized Experience Replay.
    
    Uses:
    - Double DQN target
    - Prioritized replay buffer
    - Importance sampling weighted loss
    """
    
    def __init__(self, state_dim: int, action_dim: int):
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.9998
        self.batch_size = 64
        self.target_update_freq = 250
        self.action_dim = action_dim
        
        # PER parameters
        self.alpha = 0.6
        self.beta = 0.4
        
        # Networks
        self.q_network = QNetwork(state_dim, action_dim).to(device)
        self.target_network = QNetwork(state_dim, action_dim).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()
        
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=3e-4)
        
        # Prioritized Replay Buffer
        self.memory = PrioritizedReplayBuffer(
            capacity=50000,
            alpha=self.alpha,
            beta=self.beta
        )
        
        self.steps = 0
    
    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        
        with torch.no_grad():
            q_values = self.q_network(state_tensor)
        
        return q_values.argmax(dim=1).item()
    
    def store_transition(self, state, action, reward, next_state, done):
        self.memory.push(state, action, reward, next_state, done)
    
    def update(self):
        """
        PER DQN update with importance sampling.
        """
        if len(self.memory) < self.batch_size:
            return None
        
        # Sample from prioritized buffer
        states, actions, rewards, next_states, dones, weights, indices = self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(states).to(device)
        actions = torch.LongTensor(actions).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).to(device)
        weights_tensor = torch.FloatTensor(weights).to(device)
        
        # Current Q values: Q_online(s,a)
        current_q = self.q_network(states).gather(1, actions).squeeze(1)
        
        # Target Q values: Double DQN style
        with torch.no_grad():
            # Online network selects best next action
            next_actions = self.q_network(next_states).argmax(dim=1, keepdim=True)
            
            # Target network evaluates selected action
            next_q = self.target_network(next_states).gather(1, next_actions).squeeze(1)
            
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        # TD errors for priority update
        td_errors = (target_q - current_q).detach().abs().cpu().numpy()
        
        # Weighted loss with importance sampling
        elementwise_loss = F.mse_loss(current_q, target_q, reduction="none")
        loss = (weights_tensor * elementwise_loss).mean()
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=10.0)
        self.optimizer.step()
        
        # Update priorities
        self.memory.update_priorities(indices, td_errors)
        
        # Anneal beta toward 1
        self.memory.anneal_beta()
        self.beta = self.memory.beta
        
        # Update target network
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
            self.epsilon = max(self.epsilon, self.epsilon_min)
        
        return loss.item()

In [ ]:
print("="*60)
print("TRAINING PER DQN ON LUNAR LANDER")
print("="*60)

# Reset seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

env4 = gym.make("LunarLander-v3")

state_dim = env4.observation_space.shape[0]
action_dim = env4.action_space.n

per_agent = PERDQNAgent(state_dim, action_dim)

per_scores, per_losses = train_dqn(
    env4,
    per_agent,
    num_episodes=1000,
    max_steps=1000
)

env4.close()
# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Video
video_perdqn = save_agent_video(
    env_name="LunarLander-v3",
    agent=per_agent,
    filename="perdqn.mp4",
    seed=42
)

show_video(video_perdqn, width=600)
axes[0].plot(per_scores)
axes[0].plot(np.convolve(per_scores, np.ones(50)/50, mode='valid'), 'r-', linewidth=2, label='Moving Avg')
axes[0].axhline(y=200, color='g', linestyle='--', label='Target')
axes[0].set_title('PER DQN - Scores')
axes[0].set_xlabel('Episode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(per_losses)
axes[1].set_title('PER DQN - Loss')
axes[1].set_xlabel('Episode')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nPER DQN Final Avg Score (last 50): {np.mean(per_scores[-50:]):.1f}")

show_video(video_perdqn, width=600)

# Half-Rainbow DQN (Double + Dueling + PER)

Rainbow DQN (Hessel et al., 2018) combined **6 different techniques**.
In this section we implement the 3 mentioned ones.


### Your Task
- Combine `DuelingQNetwork` architecture
- Use `PrioritizedReplayBuffer` for storage
- Implement Double DQN update logic
- Make sure priorities are updated after each learning step

In [ ]:
# =============================================================================
# Double DQN + Dueling Architecture + Prioritized Experience Replay
# =============================================================================

class HalfRainbowDQNAgent:
    """
    Half-Rainbow DQN Agent
    
    Combines:
    1. Double DQN target
    2. Dueling Q-Network architecture
    3. Prioritized Experience Replay
    """
    
    def __init__(self, state_dim: int, action_dim: int):
        # Hyperparameters
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.9998
        self.batch_size = 64
        self.target_update_freq = 500
        self.action_dim = action_dim
        
        # PER hyperparameters
        self.alpha = 0.4
        self.beta = 0.6
        
        # Dueling networks
        self.q_network = DuelingQNetwork(state_dim, action_dim).to(device)
        self.target_network = DuelingQNetwork(state_dim, action_dim).to(device)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()
        
        # Optimizer
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=3e-4)
        
        # Prioritized replay buffer
        self.memory = PrioritizedReplayBuffer(
            capacity=50000,
            alpha=self.alpha,
            beta=self.beta
        )
        
        self.steps = 0
    
    def select_action(self, state, training=True):
        """
        Epsilon-greedy action selection.
        """
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        
        with torch.no_grad():
            q_values = self.q_network(state_tensor)
        
        return q_values.argmax(dim=1).item()
    
    def store_transition(self, state, action, reward, next_state, done):
        """
        Store transition in PER buffer.
        """
        self.memory.push(state, action, reward, next_state, done)
    
    def update(self):
        """
        One Half-Rainbow update:
        - sample using PER
        - compute Double DQN target
        - use dueling networks
        - apply importance-sampling weighted loss
        - update priorities
        """
        if len(self.memory) < self.batch_size:
            return None
        
        # Sample from PER buffer
        states, actions, rewards, next_states, dones, weights, indices = self.memory.sample(self.batch_size)
        
        states = torch.FloatTensor(states).to(device)
        actions = torch.LongTensor(actions).unsqueeze(1).to(device)
        rewards = torch.FloatTensor(rewards).to(device)
        next_states = torch.FloatTensor(next_states).to(device)
        dones = torch.FloatTensor(dones).to(device)
        weights_tensor = torch.FloatTensor(weights).to(device)
        
        # Current Q-values: Q_online(s, a)
        current_q = self.q_network(states).gather(1, actions).squeeze(1)
        
        # Double DQN target
        with torch.no_grad():
            # Online dueling network selects next action
            next_actions = self.q_network(next_states).argmax(dim=1, keepdim=True)
            
            # Target dueling network evaluates that action
            next_q = self.target_network(next_states).gather(1, next_actions).squeeze(1)
            
            # Bellman target
            target_q = rewards + self.gamma * next_q * (1 - dones)
        
        # TD errors for priority update
        td_errors = (target_q - current_q).detach().abs().cpu().numpy()
        
        # Importance-sampling weighted loss
        elementwise_loss = F.mse_loss(current_q, target_q, reduction="none")
        loss = (weights_tensor * elementwise_loss).mean()
        
        # Optimize online dueling network
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=5.0)
        self.optimizer.step()
        
        # Update priorities in PER
        self.memory.update_priorities(indices, td_errors)
        
        # Anneal beta toward 1
        self.memory.anneal_beta()
        self.beta = self.memory.beta
        
        # Update target network periodically
        self.steps += 1
        if self.steps % self.target_update_freq == 0:
            self.target_network.load_state_dict(self.q_network.state_dict())
        
        # Decay epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
            self.epsilon = max(self.epsilon, self.epsilon_min)
        
        return loss.item()

In [ ]:
print("="*60)
print("TRAINING HALF-RAINBOW DQN ON LUNAR LANDER")
print("="*60)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

env5 = gym.make("LunarLander-v3")

state_dim = env5.observation_space.shape[0]
action_dim = env5.action_space.n

print(f"State dim: {state_dim}, Action dim: {action_dim}\n")

combined_agent = HalfRainbowDQNAgent(state_dim, action_dim)

combined_scores, combined_losses = train_dqn(
    env5,
    combined_agent,
    num_episodes=1000,
    max_steps=1000
)

env5.close()
# Plotfig, axes = plt.subplots(1, 2, figsize=(12, 4))
video_half_rainbow = save_agent_video(
    env_name="LunarLander-v3",
    agent=combined_agent,
    filename="half_rainbow.mp4",
    seed=42
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(combined_scores, alpha=0.5, label="Score")

if len(combined_scores) >= 50:
    moving_avg = np.convolve(combined_scores, np.ones(50) / 50, mode="valid")
    axes[0].plot(moving_avg, "r-", linewidth=2, label="Moving Avg")

axes[0].axhline(y=200, color="g", linestyle="--", label="Solved Threshold")
axes[0].set_title("Half-Rainbow DQN - Scores")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Score")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(combined_losses)
axes[1].set_title("Half-Rainbow DQN - Loss")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Loss")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nHalf-Rainbow Final Avg Score Last 50: {np.mean(combined_scores[-50:]):.1f}")

show_video(video_half_rainbow, width=600)

# Last Task: Final Comparison - All Methods

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

window = 50

# Calculate moving averages
dqn_ma = np.convolve(dqn_scores, np.ones(window)/window, mode='valid')
ddqn_ma = np.convolve(ddqn_scores, np.ones(window)/window, mode='valid')
dueling_ma = np.convolve(dueling_scores, np.ones(window)/window, mode='valid')
per_ma = np.convolve(per_scores, np.ones(window)/window, mode='valid')
combined_ma = np.convolve(combined_scores, np.ones(window)/window, mode='valid')

# Plot 1: All moving averages
axes[0, 0].plot(dqn_ma, color='blue', linewidth=2, label='DQN')
axes[0, 0].plot(ddqn_ma, color='red', linewidth=2, label='Double DQN')
axes[0, 0].plot(dueling_ma, color='green', linewidth=2, label='Dueling DQN')
axes[0, 0].plot(per_ma, color='purple', linewidth=2, label='PER DQN')
axes[0, 0].plot(combined_ma, color='orange', linewidth=2, label='Combined')
axes[0, 0].axhline(y=200, color='black', linestyle='--', label='Target (200)')
axes[0, 0].set_title('All Methods - Moving Average (50)')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Score')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Final scores bar chart
labels = ['DQN', 'Double\nDQN', 'Dueling\nDQN', 'PER\nDQN', 'Combined']
final_scores = [
    np.mean(dqn_scores[-50:]),
    np.mean(ddqn_scores[-50:]),
    np.mean(dueling_scores[-50:]),
    np.mean(per_scores[-50:]),
    np.mean(combined_scores[-50:])
]
colors = ['blue', 'red', 'green', 'purple', 'orange']

bars = axes[0, 1].bar(labels, final_scores, color=colors, alpha=0.7)
axes[0, 1].axhline(y=200, color='black', linestyle='--', label='Target')
axes[0, 1].set_title('Average Score (Last 50 Episodes)')
axes[0, 1].set_ylabel('Score')
axes[0, 1].legend()

for bar, score in zip(bars, final_scores):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, 
                    f'{score:.1f}', ha='center', fontsize=11, fontweight='bold')

# Final Summary
print("\n" + "="*70)
print("FINAL SUMMARY - ALL METHODS COMPARISON")
print("="*70)
print(f"{'Method':<20} {'Avg Score (last 50)':<25} {'Status':<15}")
print("-"*70)
print(f"{'DQN':<20} {np.mean(dqn_scores[-50:]):<25.1f} {'Basic':<15}")
print(f"{'Double DQN':<20} {np.mean(ddqn_scores[-50:]):<25.1f} {'Reduced overestimation':<15}")
print(f"{'Dueling DQN':<20} {np.mean(dueling_scores[-50:]):<25.1f} {'Better V(s) estimation':<15}")
print(f"{'PER DQN':<20} {np.mean(per_scores[-50:]):<25.1f} {'Better sampling':<15}")
print(f"{'Combined':<20} {np.mean(combined_scores[-50:]):<25.1f} {'All techniques':<15}")
print("="*70)
print(f"Target Score: 200 (solved)")